In [ ]:
library(GSEABase)
library(msigdbr)
library(Biobase)
library(genefilter)
library(limma)
library(RColorBrewer)
library(GSVA)
library(clusterProfiler)
library(stats)

library(ggplot2)
library(RColorBrewer)

require(enrichplot)
require(stats4)
library(tidyr)
library(homologene)

In [ ]:

# Basic function to convert mouse to human gene names
convertMouseGeneList <- function(x){

require("biomaRt")
human = useMart("ensembl", dataset = "hsapiens_gene_ensembl",host="https://dec2021.archive.ensembl.org/")
mouse = useMart("ensembl", dataset = "mmusculus_gene_ensembl",host="https://dec2021.archive.ensembl.org/")
genesV2 = getLDS(attributes = c("mgi_symbol"), filters = "mgi_symbol", values = x , mart = mouse, attributesL = c("hgnc_symbol"), martL = human, uniqueRows=T)
humanx <- genesV2#unique(genesV2[, 2])

# Print the first 6 genes found to the screen
print(head(humanx))
return(humanx)
}

In [ ]:
# Basic function to convert human to mouse gene names
convertHumanGeneList <- function(x){

require("biomaRt")
human = useMart("ensembl", dataset = "hsapiens_gene_ensembl",host="https://dec2021.archive.ensembl.org/")
mouse = useMart("ensembl", dataset = "mmusculus_gene_ensembl",host="https://dec2021.archive.ensembl.org/")

genesV2 = getLDS(attributes = c("hgnc_symbol"), filters = "hgnc_symbol", values = x , mart = human, attributesL = c("mgi_symbol"), martL = mouse, uniqueRows=T)

humanx <- genesV2#unique(genesV2[, 2])

# Print the first 6 genes found to the screen
print(head(humanx))
return(humanx)
}



In [ ]:
getwd()
d="/home/xili/Documents/GSEA"
geneset_name="KEGG"
subset=""
subset_file=paste0(geneset_name,subset,".gmt")
geneset <- getGmt(file.path(d
  ,subset_file ) )

In [ ]:
project_ID="pool_Myeloid"
group_by="trajecgory_new_y"
mtx="transcriptome"

# GSVA

In [ ]:
matrix = "avr"

In [ ]:
X <- read.table(file = paste0(mtx,"/",project_ID, "_DEG/", project_ID,"_",group_by,"_",matrix,".csv"),header=T, sep=",",row.names=1)

In [ ]:
HGNC <- convertMouseGeneList(row.names(X))

In [ ]:
X_humanized<-merge(X,HGNC,by.x=0,by.y = "MGI.symbol")

In [ ]:
row.names(X_humanized) <- make.unique(X_humanized$HGNC.symbol)

In [ ]:
X_humanized<-X_humanized[,2:(dim(X)[2]+1)]

In [ ]:
dim(X_humanized)

In [ ]:
sub_group = "M2_TAM"

In [ ]:
list_target1 <- c()

for (m in colnames(X_humanized)) {
  list_ <- grepl(sub_group,m, fixed=T)
  list_target1 <- append(list_target1, list_)
}

In [ ]:
X<-X_humanized#[,colnames(X_humanized)[list_target1]]

In [ ]:
meta=read.table(file = paste0(mtx,"/",project_ID, "_DEG/" ,"pool_Myeloid_meta_annotation.csv"),header=T, sep=",", row.names=1)

In [ ]:
X= as.matrix(X)

In [ ]:
es.diff <- gsva(X, geneset, kcdf="Poisson",
              method="gsva",
               mx.diff=TRUE, verbose=TRUE, 
               parallel.sz=10, min.sz=15)
head(es.diff,n=3)

In [ ]:
es.diff_ <- as.data.frame(es.diff)#@assayData[["exprs"]]
write.csv(es.diff,paste0(mtx, "/",project_ID, "_DEG/",project_ID,subset,"_",matrix,"_GSVA.csv"))
es.diff_ <- t(es.diff)
es.diff_<- t(scale(es.diff_))

In [ ]:
library(pheatmap)
library(circlize)
library(ComplexHeatmap)

In [ ]:

pdf(paste0(mtx, "/",project_ID, "_DEG/",geneset_name,"_",group_by,"_",matrix,".pdf"),width=10, height =6, pointsize = 4)
pheatmap(es.diff_
                   ,fontsize=6, cluster_cols =T, 
                   annotation_col=meta,
                   #annotation_colors = ann_colors,
          clustering_method = "ward.D2",
         color=colorRamp2(c(-1.5,0,1.5),c("royalblue","papayawhip","tomato")),
                   show_colnames =F,show_rownames =T
        )#, scale="column")
dev.off()

# GSEA

In [ ]:
X <- read.table(file = paste0(mtx,"/",project_ID, "_DEG/", group_by,".csv"),header=T, sep=",")#, row.names = 1)

In [ ]:
anno<-bitr(X$X, fromType = "SYMBOL", toType = "ENTREZID",OrgDb = "org.Mm.eg.db")
X<-merge(X,anno,by.x="X",by.y = "SYMBOL")
#write.csv(X,"149c9159vc5d2_merge.csv",row.names=FALSE,col.names=TRUE,sep=",")


In [ ]:
geneList = X[,2]
names(geneList) = as.character(X[,3])
ordered_genes = sort(geneList, decreasing = TRUE)
gene_rank <- length(ordered_genes):1
names(gene_rank) <- names(ordered_genes)

In [ ]:
#geneList
kk <- gseKEGG(gene_rank, organism = "mouse",pvalueCutoff =1,
#kk <- gseGO(gene_rank,ont = "BP", OrgDb = "org.Mm.eg.db",pvalueCutoff =1,
          minGSSize = 5, maxGSSize = 500)
#kk <- gseGO(geneList,ont = "MF", OrgDb = org.Hs.eg.db,pvalueCutoff = 0.05)
#kk <- gseGO(geneList,ont = "CC", OrgDb = org.Hs.eg.db,pvalueCutoff = 0.05)
#kk <-GSEA(geneList,TERM2GENE=H, verbose=FALSE,pvalueCutoff =1,
##         minGSSize = 5, maxGSSize = 100)
sortkk_up<-kk[order(kk$NES,decreasing=T)]
sortkk_dn<-kk[order(kk$NES,decreasing=F)]

sortkk_up$sig <- "no"
sortkk_up$sig[sortkk_up$p.adjust <= 0.5 &sortkk_up$NES >= sortkk_up$NES[5]] <- "up"
sortkk_up$sig[sortkk_up$p.adjust <= 0.5 &sortkk_up$NES <= sortkk_dn$NES[5]] <- "down"

In [ ]:
gene_set="KEGG"
pdf(file =paste0(mtx,"/",project_ID, "_DEG/",group_by,"_",gene_set,"_valcano_pos.pdf"),width=6,height=4)
p <- ggplot(data = sortkk_up, 
            aes(x = NES, 
                y = -log10(pvalue)))+
  geom_point(alpha=0.4, size=3.5, 
             aes(color=sig)) +
  scale_color_manual(values=c("blue", "grey","red"))+
  geom_vline(xintercept=c(-1,1),lty=4,col="black",lwd=0.8) +
  geom_hline(yintercept = -log10(0.1),lty=4,col="black",lwd=0.8) +
  theme_bw()
for_label <- sortkk_up %>% 
  filter(sortkk_up$sig!="no")#& '-1*log10(adj.P.Val)'> -1*log10(0.000001))
p <- p+ geom_point(size = 3, shape = 1, data = for_label) +
  ggrepel::geom_label_repel(
    aes(label =Description ),
    data = for_label,
    color="black"
  ) + labs(title = paste(group_by,"_",gene_set))
print(p)

dev.off()

In [ ]:
gene_set="KEGG"
kk@result=kk@result[order(kk@result[,"NES"]),]
kk_pNES=kk
kk_pNES@result <- subset(kk_pNES@result,NES>0)
kk_pNES@result <- subset(kk_pNES@result, pvalue<0.1)
kk_pNES@result <- kk_pNES@result[order(kk_pNES@result[,"pvalue"],decreasing=F),]
kk_pNES@result=kk_pNES@result[order(kk_pNES@result[,"NES"],decreasing=T),]
kk_nNES=kk
kk_nNES@result <- subset(kk_nNES@result,NES<0)
kk_nNES@result <- subset(kk_nNES@result, pvalue<0.1)
kk_nNES@result <- kk_nNES@result[order(kk_nNES@result[,"pvalue"],decreasing=F),]
kk_nNES@result=kk_nNES@result[order(kk_nNES@result[,"NES"],decreasing=F),]
kk_comb <- kk_pNES
kk_comb@result <- rbind(kk_pNES@result,kk_nNES@result)
nrow_kk_comb <- as.numeric(nrow(kk_comb@result))
pdf(file =paste0(mtx,"/",project_ID, "_DEG/",group_by,"_",gene_set,"_gseaplot_pos.pdf"),width=6,height=6)
gseaplot2(kk_pNES, c(1:5),
          base_size=10,rel_heights=c(3,1,1,1),subplots=1:5,ES_geom="line",
          pvalue_table=T, title=paste0(group_by,"_",gene_set))

dev.off()

In [ ]:
geneList = X[,2]
names(geneList) = as.character(X[,3])
ordered_genes = sort(geneList, decreasing = FALSE)
gene_rank <- length(ordered_genes):1
names(gene_rank) <- names(ordered_genes)

In [ ]:
#geneList
kk <- gseKEGG(gene_rank, organism = "mouse",pvalueCutoff =1,
#kk <- gseGO(gene_rank,ont = "BP", OrgDb = "org.Mm.eg.db",pvalueCutoff =1,
          minGSSize = 5, maxGSSize = 500)
#kk <- gseGO(geneList,ont = "MF", OrgDb = org.Hs.eg.db,pvalueCutoff = 0.05)
#kk <- gseGO(geneList,ont = "CC", OrgDb = org.Hs.eg.db,pvalueCutoff = 0.05)
#kk <-GSEA(geneList,TERM2GENE=H, verbose=FALSE,pvalueCutoff =1,
#          minGSSize = 5, maxGSSize = 100)
sortkk_up<-kk[order(kk$NES,decreasing=T)]
sortkk_dn<-kk[order(kk$NES,decreasing=F)]

sortkk_up$sig <- "no"
sortkk_up$sig[sortkk_up$p.adjust <= 0.5 &sortkk_up$NES >= sortkk_up$NES[5]] <- "up"
sortkk_up$sig[sortkk_up$p.adjust <= 0.5 &sortkk_up$NES <= sortkk_dn$NES[5]] <- "down"

In [ ]:
gene_set="KEGG"
pdf(file =paste0(mtx,"/",project_ID, "_DEG/",group_by,"_",gene_set,"_valcano_dn.pdf"),width=6,height=4)
p <- ggplot(data = sortkk_up, 
            aes(x = NES, 
                y = -log10(pvalue)))+
  geom_point(alpha=0.4, size=3.5, 
             aes(color=sig)) +
  scale_color_manual(values=c("blue", "grey","red"))+
  geom_vline(xintercept=c(-1,1),lty=4,col="black",lwd=0.8) +
  geom_hline(yintercept = -log10(0.1),lty=4,col="black",lwd=0.8) +
  theme_bw()
for_label <- sortkk_up %>% 
  filter(sortkk_up$sig!="no")#& '-1*log10(adj.P.Val)'> -1*log10(0.000001))
p <- p+ geom_point(size = 3, shape = 1, data = for_label) +
  ggrepel::geom_label_repel(
    aes(label =Description ),
    data = for_label,
    color="black"
  ) + labs(title = paste(group_by,"_",gene_set))
print(p)

dev.off()

In [ ]:
gene_set="KEGG"
kk@result=kk@result[order(kk@result[,"NES"]),]
kk_pNES=kk
kk_pNES@result <- subset(kk_pNES@result,NES>0)
kk_pNES@result <- subset(kk_pNES@result, pvalue<0.1)
kk_pNES@result <- kk_pNES@result[order(kk_pNES@result[,"pvalue"],decreasing=F),]
kk_pNES@result=kk_pNES@result[order(kk_pNES@result[,"NES"],decreasing=T),]
kk_nNES=kk
kk_nNES@result <- subset(kk_nNES@result,NES<0)
kk_nNES@result <- subset(kk_nNES@result, pvalue<0.1)
kk_nNES@result <- kk_nNES@result[order(kk_nNES@result[,"pvalue"],decreasing=F),]
kk_nNES@result=kk_nNES@result[order(kk_nNES@result[,"NES"],decreasing=F),]
kk_comb <- kk_pNES
kk_comb@result <- rbind(kk_pNES@result,kk_nNES@result)
nrow_kk_comb <- as.numeric(nrow(kk_comb@result))
pdf(file =paste0(mtx,"/",project_ID, "_DEG/",group_by,"_",gene_set,"_gseaplot_dn.pdf"),width=6,height=6)
gseaplot2(kk_pNES, c(1:5),
          base_size=10,rel_heights=c(3,1,1,1),subplots=1:5,ES_geom="line",
          pvalue_table=T, title=paste0(group_by,"_",gene_set))

dev.off()

In [ ]:
kk_pNES@result
